In [33]:
import cv2
import numpy as np
import os

import sys
sys.path.append('../src')

from data_handling import read_image, resize_and_crop, convert_to_grayscale, balance_brightness, reduce_noise
from geometry_analysis import detect_edges, apply_roi_mask, get_lines, filter_lines_by_length, filter_lines_by_angle
from object_analysis import run_object_detection

class SimpleDrivingController:
    def __init__(self, frame_width, frame_height):
        self.width = frame_width
        self.height = frame_height
        self.camera_center_x = self.width // 2
        
        self.steering_margin = 30 
        
        self.danger_y_line = int(self.height * 0.8) 

    def get_steering_decision(self, left_lane_x, right_lane_x):
        if left_lane_x is None or right_lane_x is None:
            return "HOLD" 
            
        lane_center_x = (left_lane_x + right_lane_x) // 2
        offset = lane_center_x - self.camera_center_x
        
        if offset > self.steering_margin:
            return "TURN RIGHT"
        elif offset < -self.steering_margin:
            return "TURN LEFT"
        else:
            return "GO STRAIGHT"

    def get_speed_decision(self, yolo_results, left_lane_x, right_lane_x):
        """
        Đưa ra quyết định phanh dựa trên Bounding Boxes của YOLO.
        yolo_results là danh sách các [x1, y1, x2, y2, conf, class_id]
        """
        # Giả sử làn đường có độ rộng mặc định nếu không nhận diện được
        safe_left = left_lane_x if left_lane_x else 0
        safe_right = right_lane_x if right_lane_x else self.width

        for box in yolo_results:
            x1, y1, x2, y2 = box[:4] # Tọa độ bounding box
            class_id = int(box[5])
            
            # Chỉ quan tâm xe cộ (0: person, 2: car, 3: motorcycle, 5: bus, 7: truck)
            if class_id not in [0, 2, 3, 5, 7]:
                continue
                
            obj_center_x = (x1 + x2) // 2
            
            # Kiểm tra 2 điều kiện: Vật ở TRONG LÀN và VƯỢT VẠCH NGUY HIỂM
            in_ego_lane = safe_left < obj_center_x < safe_right
            is_too_close = y2 > self.danger_y_line
            
            if in_ego_lane and is_too_close:
                return "BRAKE !" # Gặp nguy hiểm, phanh gấp!
                
        return "KEEP SPEED" # An toàn

    def draw_dashboard(self, frame, steering, speed):
        """Trực quan hóa quyết định lên ảnh"""
        color_speed = (0, 0, 255) if speed == "BRAKE !" else (0, 255, 0)
        color_steer = (255, 0, 0) if steering != "GO STRAIGHT" else (0, 255, 0)
        
        cv2.putText(frame, f"CMD: {speed}", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, color_speed, 2)
        cv2.putText(frame, f"STEER: {steering}", (50, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, color_steer, 2)
        
        cv2.line(frame, (0, self.danger_y_line), (self.width, self.danger_y_line), (0, 0, 255), 2)
        
        return frame

In [34]:
def get_lane_x_coordinates(lines, img_height, img_width):
    """
    Hàm phụ trợ: Phân loại đường thẳng thành lề trái/phải và ước lượng tọa độ X ở đáy ảnh.
    """
    left_x_list = []
    right_x_list = []
    
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if x2 == x1: continue # Tránh chia cho 0
            
            slope = (y2 - y1) / (x2 - x1)
            # Extrapolate (kéo dài) đường thẳng xuống đáy ảnh (y = img_height)
            # Công thức: x = x1 + (y - y1) / slope
            x_bottom = int(x1 + (img_height - y1) / slope)
            
            if slope < 0 and x_bottom < img_width // 2: # Làn trái (độ dốc âm, nằm nửa trái)
                left_x_list.append(x_bottom)
            elif slope > 0 and x_bottom > img_width // 2: # Làn phải (độ dốc dương, nằm nửa phải)
                right_x_list.append(x_bottom)
                
    # Lấy trung bình cộng các điểm cắt ở đáy ảnh
    left_lane_x = int(np.mean(left_x_list)) if left_x_list else None
    right_lane_x = int(np.mean(right_x_list)) if right_x_list else None
    
    return left_lane_x, right_lane_x

In [52]:
# 1. Khai báo đường dẫn
img_path = '../data/test-model/img3.jpg'
    
# 2. Tiền xử lý dữ liệu (Data Handling)
print("[1/5] Đang đọc và tiền xử lý ảnh...")
img = read_image(img_path)
img_cropped = resize_and_crop(img, target_width=800, target_height=600, crop_margin=50) # Đổi kích thước theo ý muốn
img_gray = convert_to_grayscale(img_cropped)
img_balanced = balance_brightness(img_gray)
img_blur = reduce_noise(img_balanced, kernel_size=5)
    
# 3. Phân tích hình học (Geometry Analysis - Hough Lines)
print("[2/5] Đang trích xuất cấu trúc làn đường...")
edges = detect_edges(img_blur, low_thresh=50, high_thresh=180)
roi_edges = apply_roi_mask(edges, upper_limit=0.4)
raw_lines = get_lines(roi_edges, threshold=60, minLineLength=80, maxLineGap=80)
    
filtered_lines = filter_lines_by_length(raw_lines)
final_lines = filter_lines_by_angle(filtered_lines, min_angle=20, max_angle=85)
    
# Tìm tọa độ X của làn trái và phải ở đáy ảnh
height, width = img_cropped.shape[:2]
left_lane_x, right_lane_x = get_lane_x_coordinates(final_lines, height, width)
    
# 4. Nhận diện vật thể (Object Detection - YOLOv8)
print("[3/5] Đang nhận diện vật thể với YOLOv8...")
yolo_result = run_object_detection(img_cropped) 
    
# Lấy dữ liệu Bounding Box một cách an toàn
boxes_data = []
if len(yolo_result.boxes) > 0:
    boxes_data = yolo_result.boxes.data.cpu().numpy() 
    
# 5. Hệ thống ra quyết định (Decision Making)
print("[4/5] Đang xử lý quyết định điều khiển...")
controller = SimpleDrivingController(frame_width=width, frame_height=height)
    
steering_cmd = controller.get_steering_decision(left_lane_x, right_lane_x)
speed_cmd = controller.get_speed_decision(boxes_data, left_lane_x, right_lane_x)
    
# 6. Trực quan hóa (Visualization)
print("[5/5] Đang vẽ kết quả trực quan...")
result_img = img_cropped.copy()
    
# Vẽ các vạch kẻ đường tìm được
if final_lines is not None:
    for line in final_lines:
        x1, y1, x2, y2 = line[0]
        cv2.line(result_img, (x1, y1), (x2, y2), (0, 255, 255), 3) # Màu vàng
            
# Vẽ Bounding Box của YOLO tự code (Bỏ qua hàm plot() của YOLO)
for box in boxes_data:
    x1, y1, x2, y2 = map(int, box[:4])
    conf = float(box[4])
    class_id = int(box[5])
        
    # Chỉ vẽ người (0) và các loại xe cộ
    if class_id in [0, 2, 3, 5, 7]: 
        # Đổi màu xanh lá nếu an toàn, màu cam nếu là đối tượng gây phanh
        # (Bạn có thể tùy chỉnh logic màu sắc ở đây nếu muốn)
        color = (255, 0, 255) # Tím mặc định
        cv2.rectangle(result_img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(result_img, f"ID:{class_id} {conf:.2f}", (x1, y1 - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            
# Vẽ bảng điều khiển (Dashboard)
result_img = controller.draw_dashboard(result_img, steering_cmd, speed_cmd)
    
# Hiển thị ảnh
cv2.imshow("ADAS Pipeline Test", result_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

save_path ='../results/Images/test-model/'
name = os.path.basename(img_path)
final_path = os.path.join(save_path, name)

os.makedirs(save_path, exist_ok=True)
cv2.imwrite(final_path, result_img)

[1/5] Đang đọc và tiền xử lý ảnh...
[2/5] Đang trích xuất cấu trúc làn đường...
[3/5] Đang nhận diện vật thể với YOLOv8...

0: 480x640 1 bus, 68.7ms
Speed: 3.0ms preprocess, 68.7ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)
[4/5] Đang xử lý quyết định điều khiển...
[5/5] Đang vẽ kết quả trực quan...


True